# EinMix: Learnable Tensor Operations

EinMix is einops' learnable layer — it combines `einsum` with a learned weight matrix. If `rearrange` is the Swiss Army knife of reshaping, EinMix is the Swiss Army knife of **linear projections**.

The core idea: any linear layer is just an einsum with a weight tensor. EinMix lets you specify *which axes* the weight operates on using the same string notation you already know from einops.

This notebook follows a **worked example → exercises** pattern. Study each worked example carefully, then solve the exercises that follow.

In [ ]:
import torch
import torch.nn as nn
from einops import rearrange, reduce, repeat
from einops.layers.torch import EinMix, Rearrange, Reduce

---
## Section 1: EinMix Basics — Linear Layers as EinMix

Every `nn.Linear` is secretly an einsum: `output[b,t] = sum_c(input[b,t,c] * weight[c,out]) + bias[out]`.

EinMix makes this explicit. You specify:
- **The pattern**: which axes exist in input and output (same notation as `rearrange`)
- **`weight_shape`**: which axes the learned weight covers
- **`bias_shape`**: which axes the learned bias covers (optional)

Axes that appear in the pattern but NOT in `weight_shape` are **batch-like** — the operation is applied independently along them.

### Worked Example 1: Standard Linear Layer as EinMix

In [ ]:
# Standard PyTorch linear layer
linear = nn.Linear(in_features=64, out_features=32)

# The SAME thing expressed as EinMix:
layer = EinMix(
    'b t c_in -> b t c_out',  # pattern: batch and time are preserved, features change
    weight_shape='c_in c_out',  # the weight connects input features to output features
    bias_shape='c_out',         # bias has one value per output feature
    c_in=64, c_out=32           # specify the sizes
)

x = torch.randn(2, 10, 64)  # batch=2, seq_len=10, features=64
out = layer(x)
print(f'Input:  {x.shape}')   # (2, 10, 64)
print(f'Output: {out.shape}')  # (2, 10, 32)

# Why is this useful? Because b and t are batch-like — the same weight matrix
# is applied independently to every (batch, time) position. This is EXACTLY
# what nn.Linear does, but EinMix makes the semantics explicit.

**Key points:**
- `b t` appear in both input and output patterns but NOT in `weight_shape` → they are batch dimensions
- `c_in` appears only on the left → it is summed over (contracted)
- `c_out` appears only on the right → it is a new output dimension
- The weight tensor has shape `(c_in, c_out)` = `(64, 32)` — exactly like `nn.Linear`

### Exercise 1.1

Create an EinMix layer that acts as a linear projection from **128 features to 64**, operating on input shape `(batch, time, 128)`. Verify the output shape.

In [ ]:
# YOUR CODE HERE
proj = ...

x = torch.randn(4, 20, 128)
out = proj(x)
print(f'Output shape: {out.shape}')  # Should be (4, 20, 64)
assert out.shape == (4, 20, 64), f'Expected (4, 20, 64), got {out.shape}'

<details>
<summary>Solution</summary>

```python
proj = EinMix(
    'b t c_in -> b t c_out',
    weight_shape='c_in c_out',
    bias_shape='c_out',
    c_in=128, c_out=64
)
```

</details>

### Exercise 1.2

Create an EinMix layer that projects each spatial position independently: input `(b, c, h, w)` with `c=16`, output `c_out=32`. The `h` and `w` dimensions should be treated as batch dims (not mixed — the same projection is applied at every spatial location).

In [ ]:
# YOUR CODE HERE
spatial_proj = ...

x = torch.randn(2, 16, 8, 8)
out = spatial_proj(x)
print(f'Output shape: {out.shape}')  # Should be (2, 32, 8, 8)
assert out.shape == (2, 32, 8, 8), f'Expected (2, 32, 8, 8), got {out.shape}'

<details>
<summary>Solution</summary>

```python
spatial_proj = EinMix(
    'b c_in h w -> b c_out h w',
    weight_shape='c_in c_out',
    bias_shape='c_out',
    c_in=16, c_out=32
)
```

`h` and `w` appear in both input and output but not in `weight_shape`, so they are batch-like. The same 16→32 projection is applied independently at every spatial position.

</details>

---
## Section 2: Mixing Spatial Dimensions (Token Mixing)

Here is the real power of EinMix: **you choose which axis to mix**.

A standard linear layer always mixes features (the last dim). But what if you want to mix *across positions*? In the MLP-Mixer architecture, this is called **token mixing** — information flows between spatial locations while each channel is processed independently.

### Worked Example 2: Token Mixing

In [ ]:
# Mix information ACROSS tokens (positions), with each channel processed independently
mixer = EinMix(
    'b t_in c -> b t_out c',   # tokens change, channels stay
    weight_shape='t_in t_out',  # weight connects input positions to output positions
    bias_shape='t_out',
    t_in=10, t_out=10
)

x = torch.randn(2, 10, 64)
out = mixer(x)
print(f'Input:  {x.shape}')   # (2, 10, 64)
print(f'Output: {out.shape}')  # (2, 10, 64)

# Compare to channel mixing (standard linear):
# Channel mixing:  weight is (c_in, c_out) — mixes features, positions are batch-like
# Token mixing:    weight is (t_in, t_out) — mixes positions, channels are batch-like
#
# This is the key insight of MLP-Mixer: alternate between these two kinds of mixing.

**Why this matters:** In a standard transformer, position-wise mixing happens through attention (O(n²) in tokens). Token mixing via EinMix is a simpler alternative — a learned weight matrix that directly maps "what each output position should attend to." It is O(n²) in parameters but O(n) in compute per channel.

### Exercise 2.1

Create a token mixer that **downsamples** from 196 tokens to 49 tokens (like going from 14x14 patches to 7x7 in a vision model). Features = 64.

In [ ]:
# YOUR CODE HERE
downsampler = ...

x = torch.randn(1, 196, 64)
out = downsampler(x)
print(f'Output shape: {out.shape}')  # Should be (1, 49, 64)
assert out.shape == (1, 49, 64), f'Expected (1, 49, 64), got {out.shape}'

<details>
<summary>Solution</summary>

```python
downsampler = EinMix(
    'b t_in c -> b t_out c',
    weight_shape='t_in t_out',
    bias_shape='t_out',
    t_in=196, t_out=49
)
```

The weight matrix has shape `(196, 49)` — each of the 49 output tokens is a learned linear combination of all 196 input tokens. Channels are batch-like, so this mixing is the same for every channel.

</details>

### Exercise 2.2

Create a "cross-dimension" mixer: input `(b, h, w, c)` with `h=8, w=8, c=32`. Mix ONLY along the height dimension, keeping `w` and `c` as batch dims. Output should have `h_out=8`.

In [ ]:
# YOUR CODE HERE
height_mixer = ...

x = torch.randn(2, 8, 8, 32)
out = height_mixer(x)
print(f'Output shape: {out.shape}')  # Should be (2, 8, 8, 32)
assert out.shape == (2, 8, 8, 32), f'Expected (2, 8, 8, 32), got {out.shape}'

<details>
<summary>Solution</summary>

```python
height_mixer = EinMix(
    'b h_in w c -> b h_out w c',
    weight_shape='h_in h_out',
    bias_shape='h_out',
    h_in=8, h_out=8
)
```

`w` and `c` are not in the weight_shape, so they are batch-like. Each (w, c) slice gets the same height-mixing independently. This is equivalent to transposing, applying a linear layer along the height axis, and transposing back — but EinMix does it in one clean expression.

</details>

---
## Section 3: Patch Embeddings with EinMix

In Vision Transformers (ViT), the first operation is splitting an image into patches and linearly embedding each patch. This is typically done with a `Conv2d(kernel_size=patch_size, stride=patch_size)` — but EinMix can express the same operation more transparently.

### Worked Example 3: ViT Patch Embedding

In [ ]:
# Input: image (b, channels, height, width)
# Output: sequence of patch embeddings (b, num_patches, embed_dim)

patch_embed = EinMix(
    'b c_in (h hp) (w wp) -> b (h w) c_out',
    #       ^^^^^^  ^^^^^^
    #  height = grid_h * patch_h,  width = grid_w * patch_w
    #
    # c_in, hp, wp are in weight_shape → contracted (mixed together)
    # h, w are batch-like → become the sequence dimension
    weight_shape='c_in hp wp c_out',
    bias_shape='c_out',
    c_in=3, hp=16, wp=16, c_out=768
)

img = torch.randn(1, 3, 224, 224)
tokens = patch_embed(img)
print(f'Input:  {img.shape}')     # (1, 3, 224, 224)
print(f'Output: {tokens.shape}')  # (1, 196, 768)
print(f'Number of patches: {224 // 16} x {224 // 16} = {(224 // 16) ** 2}')

# What happened:
# 1. (h hp) decomposes 224 into h=14 grid positions × hp=16 patch pixels
# 2. (w wp) decomposes 224 into w=14 grid positions × wp=16 patch pixels
# 3. The weight mixes c_in × hp × wp → c_out (i.e., 3×16×16=768 input values → 768 output)
# 4. h and w (grid positions) are batch-like, then flattened to (h w) = 196 tokens

### Exercise 3.1

Create a patch embedding for **CIFAR-10** images: input `(b, 3, 32, 32)`, patch size **8x8**, embedding dim **256**. What shape does the output have?

In [ ]:
# YOUR CODE HERE
cifar_patch_embed = ...

img = torch.randn(4, 3, 32, 32)
tokens = cifar_patch_embed(img)
print(f'Output shape: {tokens.shape}')
# How many patches? 32/8 = 4 per side, so 4×4 = 16 patches
assert tokens.shape == (4, 16, 256), f'Expected (4, 16, 256), got {tokens.shape}'

<details>
<summary>Solution</summary>

```python
cifar_patch_embed = EinMix(
    'b c_in (h hp) (w wp) -> b (h w) c_out',
    weight_shape='c_in hp wp c_out',
    bias_shape='c_out',
    c_in=3, hp=8, wp=8, c_out=256
)
```

Output shape: `(4, 16, 256)`. The 32x32 image is split into a 4x4 grid of 8x8 patches = 16 patches, each embedded to 256 dims.

</details>

### Exercise 3.2

Create a patch embedding that uses **4x4 patches** on a `(b, 3, 64, 64)` image with embedding dim **512**. How many tokens does this produce?

In [ ]:
# YOUR CODE HERE
patch_embed_64 = ...

img = torch.randn(2, 3, 64, 64)
tokens = patch_embed_64(img)
print(f'Output shape: {tokens.shape}')
num_tokens = (64 // 4) ** 2
print(f'Number of tokens: {num_tokens}')
assert tokens.shape == (2, num_tokens, 512), f'Expected (2, {num_tokens}, 512), got {tokens.shape}'

<details>
<summary>Solution</summary>

```python
patch_embed_64 = EinMix(
    'b c_in (h hp) (w wp) -> b (h w) c_out',
    weight_shape='c_in hp wp c_out',
    bias_shape='c_out',
    c_in=3, hp=4, wp=4, c_out=512
)
```

Output shape: `(2, 256, 512)`. 64/4 = 16 per side, so 16x16 = 256 tokens, each embedded to 512 dims. Note: smaller patches = more tokens = more compute in subsequent attention layers.

</details>

---
## Section 4: Multi-Head Attention Projections

In a standard transformer, the QKV projection involves:
1. Three separate `nn.Linear` layers (or one big one)
2. Reshape to split heads: `(b, t, c) → (b, t, h, d)`
3. Transpose for attention: `(b, t, h, d) → (b, h, t, d)`

EinMix does all three in a single operation.

### Worked Example 4: Fused QKV Projection

In [ ]:
# One EinMix replaces: 3 linear layers + reshape + transpose
qkv_proj = EinMix(
    'b t c -> qkv b h t d',
    #  c is consumed (input features)
    #  qkv, h, d are produced (3 projections × heads × head_dim)
    #  b, t are batch-like (preserved)
    weight_shape='c qkv h d',
    bias_shape='qkv h d',
    c=768, qkv=3, h=12, d=64
)

x = torch.randn(1, 196, 768)
qkv = qkv_proj(x)
print(f'Input:  {x.shape}')    # (1, 196, 768)
print(f'Output: {qkv.shape}')  # (3, 1, 12, 196, 64)

q, k, v = qkv  # unpack along first dim
print(f'Q shape: {q.shape}')   # (1, 12, 196, 64) — ready for attention!

# The weight tensor shape: (768, 3, 12, 64) = (768, 2304)
# This is exactly what 3 separate nn.Linear(768, 768) would give,
# but fused and with the head dimension already split out.

### Exercise 4.1

Create the **output projection** that reverses the head-splitting: input `(b, h, t, d)` → output `(b, t, c)`. Use `h=12, d=64, c=768`.

In [ ]:
# YOUR CODE HERE
out_proj = ...

# Simulate attention output
attn_out = torch.randn(1, 12, 196, 64)  # (b, h, t, d)
out = out_proj(attn_out)
print(f'Output shape: {out.shape}')  # Should be (1, 196, 768)
assert out.shape == (1, 196, 768), f'Expected (1, 196, 768), got {out.shape}'

<details>
<summary>Solution</summary>

```python
out_proj = EinMix(
    'b h t d -> b t c',
    weight_shape='h d c',
    bias_shape='c',
    h=12, d=64, c=768
)
```

The weight has shape `(12, 64, 768)` — it contracts over heads and head_dim to produce the output features. `b` and `t` are batch-like. This replaces: concat heads → `nn.Linear(768, 768)`.

</details>

### Exercise 4.2

Create a QKV projection with **8 heads**, `d=32`, `c=256`, for input `(b, t, 256)`. Verify the output shape.

In [ ]:
# YOUR CODE HERE
qkv_small = ...

x = torch.randn(2, 50, 256)
qkv = qkv_small(x)
print(f'Output shape: {qkv.shape}')  # Should be (3, 2, 8, 50, 32)
assert qkv.shape == (3, 2, 8, 50, 32), f'Expected (3, 2, 8, 50, 32), got {qkv.shape}'

q, k, v = qkv
print(f'Q: {q.shape}, K: {k.shape}, V: {v.shape}')

<details>
<summary>Solution</summary>

```python
qkv_small = EinMix(
    'b t c -> qkv b h t d',
    weight_shape='c qkv h d',
    bias_shape='qkv h d',
    c=256, qkv=3, h=8, d=32
)
```

Note that `h * d = 8 * 32 = 256 = c`. This is the standard convention: the total head dimension equals the model dimension. The weight shape is `(256, 3, 8, 32)` — the `c=256` input features are projected to `3 * 8 * 32 = 768` values (Q, K, V for each head).

</details>

---
## Section 5: Challenge — Build a Simple MLP-Mixer Block

The [MLP-Mixer](https://arxiv.org/abs/2105.01601) architecture alternates between:
1. **Token mixing**: mix information across spatial positions (channels are batch-like)
2. **Channel mixing**: mix information across features (positions are batch-like)

Each mixing step is: LayerNorm → Linear → GELU → Linear (with an expansion factor in the hidden dim).

### Exercise 5.1

Build a complete MLP-Mixer-style block using EinMix. The block should:
1. **Layer norm** on the input
2. **Token mixing**: mix across tokens with a hidden dim expansion factor of 4 (16 → 64 → 16)
3. **Residual** add
4. **Layer norm**
5. **Channel mixing**: mix across channels with the same expansion factor (64 → 256 → 64)
6. **Residual** add

Input shape: `(b, 16, 64)` — 16 tokens, 64 channels.

Hint: You will need two sub-networks (token mixing MLP and channel mixing MLP), each as `nn.Sequential`. The residual connections cannot go inside `nn.Sequential`, so you will need a small `nn.Module` class.

In [ ]:
# YOUR CODE HERE
class MLPMixerBlock(nn.Module):
    def __init__(self, num_tokens=16, channels=64, expansion=4):
        super().__init__()
        # Token mixing MLP: LayerNorm → EinMix (expand) → GELU → EinMix (contract)
        self.token_norm = ...
        self.token_mix = ...
        
        # Channel mixing MLP: LayerNorm → EinMix (expand) → GELU → EinMix (contract)
        self.channel_norm = ...
        self.channel_mix = ...
    
    def forward(self, x):
        # Token mixing with residual
        ...
        # Channel mixing with residual
        ...
        return x

block = MLPMixerBlock(num_tokens=16, channels=64, expansion=4)
x = torch.randn(2, 16, 64)
out = block(x)
print(f'Output shape: {out.shape}')  # Should be (2, 16, 64)
assert out.shape == (2, 16, 64), f'Expected (2, 16, 64), got {out.shape}'

# Count parameters
num_params = sum(p.numel() for p in block.parameters())
print(f'Total parameters: {num_params:,}')

<details>
<summary>Solution</summary>

```python
class MLPMixerBlock(nn.Module):
    def __init__(self, num_tokens=16, channels=64, expansion=4):
        super().__init__()
        token_hidden = num_tokens * expansion
        channel_hidden = channels * expansion
        
        # Token mixing: mix across the token dimension
        self.token_norm = nn.LayerNorm(channels)
        self.token_mix = nn.Sequential(
            EinMix('b t_in c -> b t_hid c',
                   weight_shape='t_in t_hid', bias_shape='t_hid',
                   t_in=num_tokens, t_hid=token_hidden),
            nn.GELU(),
            EinMix('b t_hid c -> b t_out c',
                   weight_shape='t_hid t_out', bias_shape='t_out',
                   t_hid=token_hidden, t_out=num_tokens),
        )
        
        # Channel mixing: mix across the channel dimension
        self.channel_norm = nn.LayerNorm(channels)
        self.channel_mix = nn.Sequential(
            EinMix('b t c_in -> b t c_hid',
                   weight_shape='c_in c_hid', bias_shape='c_hid',
                   c_in=channels, c_hid=channel_hidden),
            nn.GELU(),
            EinMix('b t c_hid -> b t c_out',
                   weight_shape='c_hid c_out', bias_shape='c_out',
                   c_hid=channel_hidden, c_out=channels),
        )
    
    def forward(self, x):
        # Token mixing with residual
        x = x + self.token_mix(self.token_norm(x))
        # Channel mixing with residual
        x = x + self.channel_mix(self.channel_norm(x))
        return x
```

Key observations:
- Token mixing weights: `(16, 64)` and `(64, 16)` — these are small because there are only 16 tokens
- Channel mixing weights: `(64, 256)` and `(256, 64)` — same as a standard 2-layer MLP
- The residual connections ensure gradient flow and allow the network to learn incremental updates
- This is the complete MLP-Mixer block — stack several of these and you have the full architecture

</details>

---
## Summary

| What you want | Pattern | weight_shape |
|---|---|---|
| Linear layer (feature mixing) | `b t c_in -> b t c_out` | `c_in c_out` |
| Token mixing | `b t_in c -> b t_out c` | `t_in t_out` |
| Patch embedding | `b c (h hp) (w wp) -> b (h w) d` | `c hp wp d` |
| QKV projection + head split | `b t c -> qkv b h t d` | `c qkv h d` |
| Output projection + head merge | `b h t d -> b t c` | `h d c` |

**The mental model**: axes in `weight_shape` are mixed together. Axes NOT in `weight_shape` are batch-like (independent). That is all there is to it.